import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [3]:
#Create Fake Data

In [ ]:
np.random.seed(42)

num_users = 500
transactions_per_user = 50

countries = ["USA", "Canada", "UK", "India", "UAE", "Singapore", "Germany"]
merchant_categories = ["groceries", "food", "gas", "electronics", "travel", "luxury", "entertainment"]

rows = []

for user_id in range(1, num_users +1):
    home_country = np.random.choice(countries, p=[0.55, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05])
    avg_spend = np.random.uniform(20, 150)

    for _ in range(transactions_per_user):
        is_fraud = np.random.choice([0,1], p =[0.96, 0.04])

        if is_fraud:
            amount = avg_spend * np.random.uniform(4, 12)
            country = np.random.choice([c for c in countries if c!= home_country])
            merchant = np.random.choice(["electronics", "travel", "luxury"])
            hour = np.random.choice([0, 1, 2, 3, 4, 23])
            is_new_country = 1
            impossible_travel = 0

        else:
            amount = np.random.normal(avg_spend, avg_spend * 0.35)
            amount = max(1, amount)
            country = home_country if np.random.rand() < 0.9 else np.random.choice(countries)
            merchant = np.random.choice(merchant_categories)
            hour = np.random.randint(7, 23)
            is_new_country = 1 if country != home_country else 0
            impossible_travel = 0


        amount_vs_avg = amount / avg_spend
        
        rows.append({
            "user_id": user_id,
            "home_country": home_country,
            "transaction_country": country,
            "amount": round(amount, 2),
            "user_avg_spend": round(avg_spend, 2),
            "amount_vs_avg": round(amount_vs_avg, 2),
            "merchant_category": merchant,
            "hour": hour,
            "is_new_country": is_new_country,
            "impossible_travel": impossible_travel,
            "is_fraud": is_fraud
        })

df = pd.DataFrame(rows)
df.head()

** This Generates Data For 500 Users, 50 Transactions Each, Totalling 25,000 Transactions. **
95% Real Transactions, 5% Being flagged as Fraudulent.
Fraud Transactions are mostly purchases that exceed average transaction, merchant category of luxury/travel/electronics at random times between 11PM and 3AM.

In [13]:
df.shape

(25000, 11)

In [14]:
df["is_fraud"].value_counts(normalize=True)

is_fraud
0    0.95956
1    0.04044
Name: proportion, dtype: float64

df.head(10)

** Training Data **

In [16]:
df_encoded = pd.get_dummies(df, columns=["merchant_category"], drop_first=True)

df_encoded.head()

,user_id,home_country,transaction_country,amount,user_avg_spend,amount_vs_avg,hour,is_new_country,impossible_travel,is_fraud,merchant_category_entertainment,merchant_category_food,merchant_category_gas,merchant_category_groceries,merchant_category_luxury,merchant_category_travel
0,1,USA,USA,87.71,143.59,0.61,17,0,0,0,False,False,True,False,False,False
1,1,USA,USA,159.62,143.59,1.11,12,0,0,0,False,False,True,False,False,False
2,1,USA,USA,155.31,143.59,1.08,11,0,0,0,False,False,False,False,False,False
3,1,USA,USA,149.52,143.59,1.04,18,0,0,0,False,False,False,False,False,True
4,1,USA,USA,21.01,143.59,0.15,22,0,0,0,False,False,False,False,False,False


In [18]:
X = df_encoded.drop(columns=[
    "user_id",
    "home_country",
    "transaction_country",
    "user_avg_spend",
    "is_fraud"
])


y = df_encoded["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4798
           1       1.00      1.00      1.00       202

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000



In [19]:
joblib.dump(model, "../models/fraud_model.pkl")
joblib.dump(list(X.columns), "../models/model_features.pkl")

['../models/model_features.pkl']

In [20]:
sample = X.iloc[0].copy()

sample["amount"] = 950
sample["amount_vs_avg"] = 8.5
sample["hour"] = 2
sample["is_new_country"] = 1
sample["impossible_travel"] = 1

sample_df = pd.DataFrame([sample])

prediction = model.predict(sample_df)[0]
risk_score = model.predict_proba(sample_df)[0][1] * 100

print("Prediction:", "Fraud" if prediction == 1 else "Legit")
print("Risk Score:", round(risk_score, 2))

Prediction: Fraud
Risk Score: 78.67


In [21]:
sample = X.iloc[0].copy()

sample["amount"] = 35
sample["amount_vs_avg"] = 1.1
sample["hour"] = 14
sample["is_new_country"] = 0
sample["impossible_travel"] = 0

sample_df = pd.DataFrame([sample])

prediction = model.predict(sample_df)[0]
risk_score = model.predict_proba(sample_df)[0][1] * 100

print("Prediction:", "Fraud" if prediction == 1 else "Legit")
print("Risk Score:", round(risk_score, 2))

Prediction: Legit
Risk Score: 0.0


In [22]:
sample = X.iloc[0].copy()

sample["amount"] = 250
sample["amount_vs_avg"] = 3.0
sample["hour"] = 23
sample["is_new_country"] = 1
sample["impossible_travel"] = 0

sample_df = pd.DataFrame([sample])

prediction = model.predict(sample_df)[0]
risk_score = model.predict_proba(sample_df)[0][1] * 100

print("Prediction:", "Fraud" if prediction == 1 else "Legit")
print("Risk Score:", round(risk_score, 2))

Prediction: Legit
Risk Score: 12.67
